# Laboration 5: Regression – från data till modell

I den här laborationen gör du samma två steg som på föreläsning 5:

**Del 1 – För hand.** Du bygger en egen modell som gissar bostadspriser, mäter hur mycket den missar och låter en `for`-loop leta upp de bästa parametrarna.

**Del 2 – Med scikit-learn.** Du gör samma sak med ett färdigt verktyg och testar modellen på bostäder som den inte har tränats på.

Vi använder ett syntetiskt bostadsdataset där målet är att förutsäga ett bostadspris.

### Efter laborationen ska du kunna
- förklara vad en modell, parametrar och träning är – och göra det själv med en funktion och en `for`-loop,
- beräkna MAE (medelabsolutfelet) själv och tolka det,
- skilja på **features (`X`)** och **target (`y`)**,
- dela data i tränings- och testdata,
- träna en linjär regressionsmodell med scikit-learn,
- göra förutsägelser på ny data och utvärdera dem,
- jämföra modellen med en enkel baseline,
- undersöka om fler features förbättrar modellen.

## 0. Skapa datasetet

Kör cellen nedan. Den skapar samma typ av bostadsdata som vi använde på föreläsningen. Du behöver inte ändra koden.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

rng = np.random.default_rng(42)
n = 140

housing = pd.DataFrame({
    "size_m2": rng.uniform(35, 160, n),
    "rooms": rng.integers(1, 6, n),
    "age_years": rng.integers(0, 80, n),
})

housing["price_ksek"] = (
    500
    + 38 * housing["size_m2"]
    + 160 * housing["rooms"]
    - 7 * housing["age_years"]
    + rng.normal(0, 350, n)
)

housing.head()

## Uppgift 1: Lär känna datan

1. Ta reda på hur många rader och kolumner datasetet innehåller.
2. Skriv ut enkel beskrivande statistik för kolumnerna.
3. Gör ett scatterplot med `size_m2` på x-axeln och `price_ksek` på y-axeln.
4. Skriv 1–2 meningar: verkar bostadsstorlek kunna vara en användbar feature för att förutsäga pris?

In [ ]:
# Skriv din kod här

### Din tolkning

*Skriv 1–2 meningar här.*

# Del 1: En modell för hand

## Uppgift 2: Din egen modell

På föreläsningen skrev vi en modell som en funktion:

**pris ≈ lutning × storlek + konstant**

### 2a. Skriv modellen

Skriv klart funktionen `linjar_modell` så att den returnerar `lutning * storlek + konstant`.

Kontrollera att den gissar **4050** (tusen kr) för en bostad på 90 m² med lutning 45 och konstant 0.

In [ ]:
def linjar_modell(storlek, lutning, konstant):
    # Skriv din kod här
    ...


linjar_modell(90, lutning=45, konstant=0)

När din funktion fungerar: kör cellen nedan. Den ger dig en hjälpfunktion som ritar datan tillsammans med en linje. Du behöver inte ändra koden.

In [ ]:
def rita_linje(lutning, konstant):
    plt.scatter(housing["size_m2"], housing["price_ksek"], alpha=0.6)
    storlekar = np.linspace(30, 165, 100)
    plt.plot(storlekar, linjar_modell(storlekar, lutning, konstant), color="red")
    plt.xlabel("Storlek (m²)")
    plt.ylabel("Pris (tusen kr)")
    plt.title(f"lutning = {lutning}, konstant = {konstant}")
    plt.show()


rita_linje(lutning=45, konstant=0)

### 2b. Mät hur mycket modellen missar

Skriv funktionen `medelfel(lutning, konstant)`. Den ska:

1. räkna ut modellens gissning för **alla** bostäder (tips: `housing["size_m2"]`),
2. räkna ut felet: gissning − verkligt pris,
3. returnera medelvärdet av felens **absolutbelopp** – alltså MAE.

Tips: `.abs()` och `.mean()` fungerar på en hel kolumn.

Kontroll: `medelfel(45, 0)` ska bli ungefär **425**.

In [ ]:
def medelfel(lutning, konstant):
    # Skriv din kod här
    ...


medelfel(45, 0)

**Fråga:** Varför använder vi absolutbeloppet? Vad skulle hända om vi bara tog medelvärdet av felen?

*Skriv ditt svar här.*

### 2c. Prova för hand

Prova minst tre egna kombinationer av `lutning` och `konstant`. Använd `rita_linje` och `medelfel`.

Vilken kombination gav lägst medelfel?

In [ ]:
# Prova dina kombinationer här

### 2d. Låt datorn prova – träning

Håll konstanten fast på **500** och låt en `for`-loop prova alla lutningar från 20 till 60.

1. Spara varje lutning och dess medelfel i två listor.
2. Gör en DataFrame `kandidater` med kolumnerna `lutning` och `medelfel`.
3. Rita medelfel mot lutning.
4. Använd `sort_values` för att hitta den lutning som ger lägst medelfel.

In [ ]:
lutningar = []
medelfelen = []

for lutning in range(20, 61):
    # Skriv din kod här
    ...

# Skapa DataFrame, rita och hitta den bästa lutningen här

**Fråga:** Förklara med egna ord vad det betyder att *träna* en modell. Använd orden *parametrar* och *fel*.

*Skriv ditt svar här.*

### 2e. Frivilligt: båda parametrarna

Använd en loop inuti en loop för att prova alla kombinationer av lutningar `range(20, 61)` och konstanter `range(-500, 2001, 50)`. Vilken kombination blir bäst? Spara den bästa lutningen och konstanten – du får användning för dem i uppgift 5.

In [ ]:
# Frivillig kod här

# Del 2: Med scikit-learn

Nu gör vi samma sak med ett färdigt verktyg. Scikit-learn räknar fram den bästa linjen direkt i stället för att prova sig fram, och vi testar modellen på bostäder som den inte har tränats på.

## Uppgift 3: Välj feature och target

Precis som i del 1 börjar vi med en modell som bara får använda bostadens storlek. Storleken är vår **feature** och priset vår **target**.

Skapa:

- `X` – en **DataFrame** som endast innehåller kolumnen `size_m2`
- `y` – en **Series** som innehåller `price_ksek`

Kontrollera sedan typerna och storleken (`shape`) på `X` och `y`.

> Kom ihåg att scikit-learn vanligtvis vill att `X` ska vara en tabell, även när vi bara använder en feature.

In [ ]:
# Skapa X och y här

## Uppgift 4: Dela i träning och test

Använd `train_test_split` och lägg **25 %** av datan i testmängden.

Använd `random_state=42` så att alla får samma uppdelning.

Skapa variablerna:

- `X_train`
- `X_test`
- `y_train`
- `y_test`

Skriv sedan ut hur många observationer som finns i träning respektive test.

In [ ]:
from sklearn.model_selection import train_test_split

# Dela datan här

## Uppgift 5: Träna en modell med scikit-learn

1. Skapa en `LinearRegression`.
2. Träna den med `fit()` på **träningsdatan**.
3. Skriv ut den inlärda lutningen (`model.coef_[0]`) och konstanten (`model.intercept_`).
4. Använd `predict()` på `X_test`.
5. Spara förutsägelserna i variabeln `y_pred`.
6. Skapa en DataFrame `results` med två kolumner:
   - `actual` – facit (`y_test`)
   - `predicted` – modellens förutsägelse
7. Visa de första tio raderna.

In [ ]:
from sklearn.linear_model import LinearRegression

# Skapa, träna och använd modellen här

**Fråga:** Jämför lutningen och konstanten från scikit-learn med de bästa värdena du hittade i uppgift 2. Ligger de nära varandra? Varför blir de inte exakt lika?

*Tips: vilken data tränades modellen på, och hur noggrant provade din loop?*

*Skriv ditt svar här.*

### Visualisera prediction mot ground truth

Gör ett scatterplot med verkligt pris på x-axeln och modellens prediction på y-axeln.

Lägg gärna till diagonalen `y = x`. Ju närmare punkterna ligger diagonalen, desto bättre stämmer prediction och facit överens.

In [ ]:
# Gör diagrammet här

## Uppgift 6: Hur bra är modellen?

Beräkna modellens **Mean Absolute Error (MAE)** på testdatan.

Spara resultatet i variabeln `mae_one_feature`.

Räkna sedan ut MAE **själv**, på samma sätt som i din funktion `medelfel` (med `.abs()` och `.mean()`), och kontrollera att du får samma värde som `mean_absolute_error`.

Kom ihåg att target mäts i **tusen kronor**, så MAE har samma enhet.

In [ ]:
from sklearn.metrics import mean_absolute_error

# Beräkna MAE här

### Tolka MAE

Skriv en mening som förklarar vad ditt MAE betyder i det här problemet.

Exempel på form: *"Modellen missar bostadspriset med ungefär ... i genomsnitt."*

*Skriv din tolkning här.*

## Uppgift 7: Jämför med en baseline

En modell är inte automatiskt användbar bara för att den ger ett tal.

Skapa därför en enkel baseline som **alltid gissar medelpriset i träningsdatan**.

1. Skapa en array `baseline_pred` med ett värde för varje observation i testdatan.
2. Beräkna dess MAE och spara i `baseline_mae`.
3. Jämför `baseline_mae` med `mae_one_feature`.

**Fråga:** Är den linjära modellen bättre än baseline?

In [ ]:
# Skapa och utvärdera baseline här

### Slutsats

*Skriv 1–2 meningar här.*

## Uppgift 8: Fler features

Nu ska modellen även få veta:

- bostadens storlek (`size_m2`)
- antal rum (`rooms`)
- bostadens ålder (`age_years`)

Gör om samma arbetsflöde:

1. skapa ett nytt `X_multi`,
2. dela i train/test med samma inställningar som tidigare,
3. träna en ny `LinearRegression`,
4. gör predictions,
5. beräkna MAE och spara det i `mae_multi`.

Jämför med modellen som bara använde bostadsstorlek.

In [ ]:
# Träna och utvärdera modellen med tre features här

### Jämför modellerna

Skriv ut:

- `baseline_mae`
- `mae_one_feature`
- `mae_multi`

Vilken modell är bäst? Hur ser du det?

In [ ]:
# Skriv ut de tre resultaten här

*Skriv din slutsats här.*

## Frivillig utmaning A: Kan en rak linje beskriva vilket samband som helst?

Kör cellen nedan. Den skapar data med ett tydligt **böjt** samband.

In [ ]:
x_curve = np.linspace(-4, 4, 100)
y_curve = x_curve**2 + rng.normal(0, 1.5, 100)
curve = pd.DataFrame({"x": x_curve, "y": y_curve})

curve.plot.scatter(x="x", y="y")
plt.title("Ett icke-linjärt samband")
plt.show()

Träna en `LinearRegression` med `x` som feature och `y` som target.

Lägg modellens prediction i kolumnen `prediction` och rita både data och prediction i samma diagram.

**Fråga:** Varför blir modellen dålig trots att den får många datapunkter att träna på?

In [ ]:
# Träna modellen och visualisera resultatet här

*Skriv din förklaring här.*

## Frivillig utmaning B

Undersök själv en av följande frågor:

- Vad händer med resultatet om du ändrar `test_size`?
- Vad händer om du tar bort en av de tre features i fler-features-modellen?
- Vilken av de tre features verkar viktigast för modellen? Undersök `model.coef_` och försök tolka resultatet.

Dokumentera kort vad du testade och vad du observerade.

In [ ]:
# Frivillig kod här

## Sammanfattning

I den här laborationen har du först byggt och tränat en modell för hand:

**modell (funktion) → felmått (MAE) → träning (loop som letar bästa parametrar)**

och sedan gjort hela kedjan med scikit-learn:

**DataFrame → X/y → train/test → fit → predict → MAE → baseline → jämföra modeller**

I nästa laboration byter vi från **regression** till **klassificering** och undersöker dessutom vad som händer när en modell blir för komplex.